In [2]:
# =====================================================
# unity_communication_server.py - Python to Unity Server
# Socket communication for VR integration
# =====================================================

import socket
import json
import threading
import time
from datetime import datetime
from typing import Dict, Any, List
import os
import shutil


class UnityCommunicationServer:
    """Python server for communication with Unity VR application"""

    def __init__(self, host='127.0.0.1', port=8080):
        self.host = host
        self.port = port
        self.server_socket = None
        self.client_socket = None
        self.is_running = False
        self.request_count = 0
        self.logs = []
        print(f"[Server] Initialized on {host}:{port}")

    def start_server(self):
        """Start the socket server"""
        try:
            self.server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.server_socket.bind((self.host, self.port))
            self.server_socket.listen(1)
            self.is_running = True

            print(f"[Server] Running at http://{self.host}:{self.port}/")
            print("[Server] Waiting for Unity connection...")

            while self.is_running:
                try:
                    self.client_socket, addr = self.server_socket.accept()
                    print(f"[Server] Unity connected from {addr}")
                    self._handle_client()
                except Exception as e:
                    if self.is_running:
                        print(f"[Server] Connection error: {e}")
        except Exception as e:
            print(f"[Server] Failed to start: {e}")

    def _handle_client(self):
        """Handle client requests"""
        while self.is_running:
            try:
                data = self.client_socket.recv(4096)
                if not data:
                    break

                self.request_count += 1
                request = json.loads(data.decode('utf-8'))
                response = self._process_request(request)

                # Log request
                self.logs.append({
                    "timestamp": datetime.now().isoformat(),
                    "request": request,
                    "response": response
                })

                self.client_socket.send(json.dumps(response).encode('utf-8'))

            except Exception as e:
                print(f"[Server] Client error: {e}")
                break

        if self.client_socket:
            self.client_socket.close()

    def _process_request(self, request: Dict) -> Dict:
        """Process Unity client requests"""
        action = request.get('action', '')

        if action == 'get_skills':
            return self._get_skills_response()

        elif action == 'evaluate_rule':
            input_data = request.get('data', {})
            return self._evaluate_rule_response(input_data)

        elif action == 'chat':
            user_input = request.get('input', '')
            return self._chat_response(user_input)

        elif action == 'get_scenario':
            return self._get_scenario_response()

        elif action == 'ping':
            return {"status": "success", "message": "pong", "timestamp": datetime.now().isoformat()}

        else:
            return {"status": "error", "message": f"Unknown action: {action}"}

    def _get_skills_response(self) -> Dict:
        """Send skills data to Unity"""
        try:
            # Try to load from Knowledge Graph
            kg_paths = ["knowledge_graph.json",
                        "/content/knowledge_graph.json",
                        "SurvivalVR_Outputs/knowledge_graph.json"]

            kg_data = None
            for path in kg_paths:
                try:
                    with open(path, 'r') as f:
                        kg_data = json.load(f)
                        break
                except:
                    continue

            if kg_data:
                skills = []
                for node in kg_data.get("nodes", []):
                    if node.get("type") == "skill":
                        skills.append({
                            "id": node.get("id", ""),
                            "name": node.get("name", ""),
                            "category": node.get("category", ""),
                            "confidence": node.get("confidence", 1.0)
                        })

                return {
                    "status": "success",
                    "data": {
                        "skills": skills[:20],
                        "total": len(skills),
                        "timestamp": datetime.now().isoformat()
                    }
                }
            else:
                # Fallback sample data
                return {
                    "status": "success",
                    "data": {
                        "skills": [
                            {"id": "S1", "name": "Current Direction Reading", "category": "navigation", "confidence": 0.95},
                            {"id": "S2", "name": "Fish Oil Fire Making", "category": "survival", "confidence": 0.92},
                            {"id": "S3", "name": "Cyclone Warning Recognition", "category": "emergency", "confidence": 0.90}
                        ],
                        "total": 3,
                        "timestamp": datetime.now().isoformat()
                    }
                }
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def _evaluate_rule_response(self, input_data: Dict) -> Dict:
        """Evaluate rules and return results"""
        try:
            # Import rule engine
            from rule_engine import SurvivalRuleEngine

            # Try to load KG
            kg_paths = ["knowledge_graph.json",
                        "/content/knowledge_graph.json",
                        "SurvivalVR_Outputs/knowledge_graph.json"]

            kg_data = None
            for path in kg_paths:
                try:
                    with open(path, 'r') as f:
                        kg_data = json.load(f)
                        break
                except:
                    continue

            engine = SurvivalRuleEngine(kg_data)
            results = engine.evaluate(input_data)
            triggered = [r for r in results if r.get("status") == "triggered"]

            return {
                "status": "success",
                "data": {
                    "rules_triggered": len(triggered),
                    "results": triggered[:10],
                    "input_data": input_data,
                    "timestamp": datetime.now().isoformat()
                }
            }
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def _chat_response(self, user_input: str) -> Dict:
        """Send chatbot response"""
        try:
            from chatbot_rules import SurvivalChatbot
            chatbot = SurvivalChatbot()
            response = chatbot.get_response(user_input)
            return {
                "status": "success",
                "data": {
                    "response": response,
                    "timestamp": datetime.now().isoformat()
                }
            }
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def _get_scenario_response(self) -> Dict:
        """Generate VR scenario data"""
        scenario = {
            "name": "Cyclone Survival Training",
            "difficulty": "Medium",
            "skills_required": ["Cloud Reading", "Safe Positioning", "Emergency Response"],
            "description": "Practice survival skills during a cyclone emergency.",
            "environment": {
                "wind_speed": 65,
                "wave_height": 4.2,
                "cloud_type": "cumulonimbus",
                "visibility": "low",
                "pressure": 985
            },
            "learning_objectives": [
                "Identify cyclone warning signs using cloud patterns",
                "Take safe anchoring position",
                "Use emergency communication protocols",
                "Preserve energy during storm"
            ],
            "scoring": {
                "base_score": 0,
                "max_score": 100,
                "passing_score": 60
            }
        }
        return {
            "status": "success",
            "data": scenario
        }

    def stop_server(self):
        """Stop the server"""
        self.is_running = False
        if self.server_socket:
            self.server_socket.close()
        print("[Server] Stopped")

    def get_logs(self) -> List:
        """Return request logs"""
        return self.logs

    def export_logs(self, filename="server_logs.json"):
        """Export logs to JSON"""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.logs, f, indent=2, ensure_ascii=False)
        print(f"[Server] Logs exported to {filename}")
        return self.logs


# =====================================================
# RUN THE CODE
# =====================================================

if __name__ == "__main__":

    print("\n" + "="*70)
    print("SURVIVALVR-BD UNITY COMMUNICATION SERVER")
    print("Python Server for Unity VR Integration")
    print("="*70 + "\n")

    # Mount Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully!")
        DRIVE_PATH = "/content/drive/MyDrive/"
    except:
        print("Not in Colab. Saving locally.")
        DRIVE_PATH = "./"

    # Create output folder
    FOLDER_NAME = "SurvivalVR_Outputs"
    full_folder_path = os.path.join(DRIVE_PATH, FOLDER_NAME)
    os.makedirs(full_folder_path, exist_ok=True)
    print(f"Output folder: {full_folder_path}\n")

    # Test server without actually running (for Colab)
    print("="*70)
    print("TESTING SERVER FUNCTIONS")
    print("="*70)

    server = UnityCommunicationServer(host='127.0.0.1', port=8080)

    # Test ping
    result = server._process_request({"action": "ping"})
    print(f"Ping test: {result}")

    # Test get_skills
    result = server._process_request({"action": "get_skills"})
    print(f"Get skills test: {result.get('status')}")
    if result.get('status') == 'success':
        print(f"  Skills found: {len(result.get('data', {}).get('skills', []))}")

    # Test chat
    result = server._process_request({"action": "chat", "input": "What's the weather?"})
    print(f"Chat test: {result.get('status')}")
    if result.get('status') == 'success':
        print(f"  Response: {result.get('data', {}).get('response', '')[:60]}...")

    # Export logs
    server.export_logs("server_logs.json")

    # Copy to Drive
    try:
        shutil.copy("server_logs.json", full_folder_path)
        print(f"\nLogs exported to: {full_folder_path}")
    except:
        pass

    print("\n" + "="*70)
    print("SERVER TEST COMPLETE!")
    print("\nTo run actual server in Colab:")
    print("  server = UnityCommunicationServer()")
    print("  server.start_server()")
    print("="*70)


SURVIVALVR-BD UNITY COMMUNICATION SERVER
Python Server for Unity VR Integration

Mounted at /content/drive
Google Drive mounted successfully!
Output folder: /content/drive/MyDrive/SurvivalVR_Outputs

TESTING SERVER FUNCTIONS
[Server] Initialized on 127.0.0.1:8080
Ping test: {'status': 'success', 'message': 'pong', 'timestamp': '2026-07-01T05:58:22.530940'}
Get skills test: success
  Skills found: 3
Chat test: error
[Server] Logs exported to server_logs.json

Logs exported to: /content/drive/MyDrive/SurvivalVR_Outputs

SERVER TEST COMPLETE!

To run actual server in Colab:
  server = UnityCommunicationServer()
  server.start_server()
